In [5]:
import sys
sys.path.append("../../")
from pathlib import Path
from html import escape
import pandas as pd
from IPython.display import HTML, display

from retrieval.product_retriever import search_product_hybrid
from evaluation.eval import product_load_tests, evaluate_retrieval, context_relevance
from evaluation.classes import ContextRelevanceWrapper
from evaluation.reporting import (
    build_eval_question_report,
    build_retrieval_doc_rows,
    display_eval_question_report,
    export_retrieval_report_to_markdown,
    get_node_text,
)

In [6]:
def retrieving(test, top_k):
    context = search_product_hybrid(test.question, top_k=top_k)
    nodes = getattr(context, "nodes", context)
    return nodes

In [7]:
async def evaluate_all_retrieval(limit=None, top_k=5, markdown_path=None):
    
    tests = product_load_tests()
    selected_tests = tests[:limit] if limit is not None else tests
    rows = []
    question_reports = []

    for index, test in enumerate(selected_tests, start=1):
        nodes = retrieving(test, top_k)
        
        metrics = evaluate_retrieval(test, nodes, k=top_k)
        
        result = await context_relevance(ContextRelevanceWrapper(question=test.question, contexts=[get_node_text(item) for item in nodes]))
        metrics["context_relevance"] = result
        rows.append(metrics)
        filtered_metrics = {
            key: value
            for key, value in metrics.items()
            if key.startswith("precision") or key.startswith("recall") or key == "mrr" or key.startswith("ndcg") or key == "context_relevance"
        }
        doc_rows = build_retrieval_doc_rows(nodes, test.relevant_docs)
        display_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            metrics=filtered_metrics,
            text_blocks=[],
            doc_rows=doc_rows,
            index=index,
            total=len(selected_tests),
            metric_columns=4,
        )
        question_report = build_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            retrieved_docs=metrics["retrieved_docs"],
            doc_rows=doc_rows,
            extra_fields={"metrics": filtered_metrics},
            index=index,
            total=len(selected_tests),
        )
        question_reports.append(question_report)

    
    report_df = pd.DataFrame(rows)
    metric_columns = [column for column in report_df.columns if column.startswith(("precision", "recall", "ndcg")) or column in {"mrr", "context_relevance"}]
    summary_df = report_df[metric_columns].mean().to_frame(name="mean").T.round(3)

    display(HTML("<h2 style='margin:32px 0 12px;color:#24292f;'>Final metrics</h2>"))
    display(summary_df)

    if markdown_path is not None:
        saved_path = export_retrieval_report_to_markdown(
            report_df,
            question_reports,
            Path(markdown_path),
            notebook_path="notebooks/product/06_product_hybrid_retrieval.ipynb",
        )
        display(HTML(f"<p style='margin:12px 0 0;color:#1a7f37;'>Markdown report saved to: <code>{escape(str(saved_path))}</code></p>"))

    return report_df

In [8]:
await evaluate_all_retrieval(markdown_path="../../reports/product_hybrid_retrieving_evaluation.md", top_k=30, limit=50)

,rank,hit,section_id,title,score,preview
0,1,no,16279046,-,0.7403,Product: 20Dresses Women Blue Trousers Brand: ...
1,2,no,14220194,-,0.6205,Product: DressBerry Women Burgundy Trousers Br...
2,3,no,13523706,-,0.5000,Product: Varanga Women Black & White Geometric...
3,4,no,13647622,-,0.4860,Product: Varanga Women Pink & White Leheriya P...
4,5,no,13769372,-,0.4852,Product: Varanga Classic Black and White Bandh...
5,6,no,11421530,-,0.4290,Product: Ginni Arora Label Women Red Slim Fit ...
6,7,no,17393956,-,0.4245,Product: Fabindia Women Beige & Red Striped Co...
7,8,no,16044098,-,0.4189,Product: SASSAFRAS Women Blue Ribbed Trousers ...
8,9,no,18770032,-,0.3983,Product: Trendyol Women Black High-Rise Trouse...
9,10,no,17378344,-,0.3859,Product: Saffron Threads Women Black Original ...


,rank,hit,section_id,title,score,preview
0,1,yes,11166548,-,1.0000,Product: Athena Women Burgundy Solid Tailored ...
1,2,no,16752820,-,0.3588,Product: armure Women Brown & White Checked Su...
2,3,no,16154596,-,0.3553,Product: SHAYE Women Black Solid Suede Kamala ...
3,4,no,16565622,-,0.3483,Product: Latin Quarters Women Maroon Suede Tai...
4,5,no,10856152,-,0.3282,Product: SASSAFRAS Women Burgundy Solid Tailor...
5,6,no,16420446,-,0.3167,Product: Harpa Women Red Suede Tailored Jacket...
6,7,no,10758214,-,0.3161,Product: Athena Women Burgundy Solid Pullover ...
7,8,no,12086086,-,0.3049,Product: Athena Women Burgundy Solid Pencil Mi...
8,9,no,12402376,-,0.3043,Product: Belle Fille Women Burgundy Solid Asym...
9,10,no,11634534,-,0.2999,Product: Athena Women Burgundy Solid Basic Jum...


,rank,hit,section_id,title,score,preview
0,1,yes,18921114,-,1.0000,Product: InWeave Women Red Printed A-Line Skir...
1,2,no,17168254,-,0.9423,Product: InWeave Women Red Printed Pure Cotton...
2,3,no,15322490,-,0.7090,Product: InWeave Women Black & Yellow Lahariya...
3,4,no,15447990,-,0.6932,Product: InWeave Women Bright Orange Printed T...
4,5,no,17168250,-,0.6841,Product: InWeave Women Blue Printed Pure Cotto...
5,6,no,15760840,-,0.6791,Product: InWeave Red & Gold-Toned Regular Crop...
6,7,no,15542594,-,0.6788,Product: InWeave Women Mustard Yellow & White ...
7,8,no,16524698,-,0.6162,Product: InWeave Women Green Printed Co-Ords B...
8,9,no,17355370,-,0.5421,Product: InWeave Women Elegant Charcoal Solid ...
9,10,no,18922178,-,0.4905,Product: InWeave Women Red Flared Palazzos Bra...


,rank,hit,section_id,title,score,preview
0,1,no,10451734,-,0.7541,Product: WEAVERS VILLA Women Mustard Yellow & ...
1,2,no,18122478,-,0.6697,Product: PERFECTBLUE Mustard Yellow & Fuchsia ...
2,3,no,19218994,-,0.5824,Product: Mitera Mustard & Red Woven Design Zar...
3,4,no,17663018,-,0.5580,Product: Dupatta Bazaar Mustard Yellow Ethnic ...
4,5,no,16719554,-,0.5474,Product: Banarasi Style Mustard Woven Design D...
5,6,no,11117288,-,0.5231,Product: Mitera Mustard & Pink Silk Blend Wove...
6,7,no,18135092,-,0.5223,Product: Mitera Mustard Yellow Ethnic Motifs Z...
7,8,no,17365722,-,0.4817,Product: I Saw It First Women Beige Striped Hi...
8,9,no,18141434,-,0.4767,Product: MANOHARI Mustard & Gold-Toned Woven D...
9,10,no,14159320,-,0.4766,Product: I AM FOR YOU Women Yellow & White Emb...


,rank,hit,section_id,title,score,preview
0,1,no,14006764,-,0.5000,Product: Saree mall Floral Silk Blend Saree wi...
1,2,no,16124612,-,0.5000,Product: MANGO Black Self Design Regular Top B...
2,3,no,17607938,-,0.4349,Product: MANGO Women Coral Red Solid Double-Br...
3,4,no,18321088,-,0.4144,Product: HRITIKA Grey & Red Saree Brand: HRITI...
4,5,no,18877470,-,0.4058,Product: HRITIKA Red Satin Saree Brand: HRITIK...
5,6,no,15274016,-,0.3915,Product: MANGO Women Black Pullover Brand: MAN...
6,7,no,19165458,-,0.3908,Product: HRITIKA Women Pink Georgette Floral C...
7,8,no,17336142,-,0.3896,Product: UNDER ARMOUR Women Pink Loose Fit Tra...
8,9,no,18224860,-,0.3876,Product: HRITIKA Black Sequinned Saree Brand: ...
9,10,no,17808618,-,0.3834,Product: MANGO Off White Satin Finish Extended...


...

,rank,hit,section_id,title,score,preview
0,1,yes,16835204,-,0.7309,Product: Roadster Women Grey Slim Flared High-...
1,2,yes,16835178,-,0.7144,Product: Roadster Women Grey Slim Flared High-...
2,3,yes,16835166,-,0.6851,Product: The Roadster Lifestyle Co Women Grey ...
3,4,yes,13339544,-,0.6280,Product: The Roadster Lifestyle Co Women Grey ...
4,5,yes,8319213,-,0.6050,Product: Roadster Women Grey Skinny Fit Mid-Ri...
5,6,no,14954602,-,0.5422,Product: Roadster Women Alluring Grey High-Ris...
6,7,yes,14954872,-,0.5386,Product: Roadster Women Grey Slouchy Fit Mid-R...
7,8,no,14536020,-,0.5363,Product: Roadster Woman Beautiful Grey Solid D...
8,9,no,14535748,-,0.5341,Product: Roadster Women Alluring Grey Washed D...
9,10,no,16172382,-,0.5000,Product: NEUDIS Red Cowl Neck Satin Regular To...


,rank,hit,section_id,title,score,preview
0,1,no,16931960,-,1.0000,Product: all about you Teal Blue Self Design K...
1,2,no,4374524,-,0.7053,Product: RARE Women Green Self Design Peplum T...
2,3,no,5617740,-,0.6865,Product: plusS Sea Green Self Design Peplum To...
3,4,yes,16504028,-,0.6540,Product: ANVI Be Yourself Women Stunning Blue ...
4,5,no,18514120,-,0.5930,Product: Paralians Turquoise Blue Peplum Top B...
5,6,no,14641744,-,0.5881,Product: FabAlley Blue Puff Sleeve Chambray Pe...
6,7,no,13382002,-,0.5663,Product: plusS Women Black Self-Design Smocked...
7,8,no,11607414,-,0.5359,Product: Bhama Couture Women Blue Printed Pepl...
8,9,no,11607374,-,0.5075,Product: Bhama Couture Women Blue Printed Pepl...
9,10,no,17132282,-,0.5065,Product: Athena Women Pretty Pink Self-Design ...


,rank,hit,section_id,title,score,preview
0,1,yes,16533076,-,0.9950,Product: Madame Women Peach-Coloured Self Desi...
1,2,no,18948340,-,0.8105,Product: Madame Peach-Coloured Cuffed Sleeves ...
2,3,no,14556346,-,0.8037,Product: Madame Women White Solid Fuzzy Pullov...
3,4,no,15756110,-,0.6470,Product: Madame Women Pink Pullover Woolen Swe...
4,5,no,16512732,-,0.6163,Product: Madame Women Beige Self Design Pullov...
5,6,no,15997770,-,0.5886,Product: Madame Women Yellow & White Checked P...
6,7,no,16566862,-,0.4745,Product: Madame Women Mauve Lightweight Puffer...
7,8,no,15964118,-,0.4597,Product: Madame Women Charcoal Front-Open Swea...
8,9,no,16512720,-,0.4022,Product: Madame Women Grey Solid Cardigan Swea...
9,10,no,10891532,-,0.4019,Product: Alsace Lorraine Paris Women Peach-Col...


,precision,recall,mrr,ndcg,context_relevance
mean,0.081,0.745,0.597,0.617,0.895


,question,relevant_docs,retrieved_docs,precision,recall,mrr,ndcg,context_relevance
0,Show me Regular trousers within a budget of 2000.,"[17817750, 18770002, 18769968, 18769944, 18769...","[16279046, 14220194, 13523706, 13647622, 13769...",0.000000,0.000,0.000000,0.000000,0.75
1,"Can you find something similar to ""Athena Wome...",[11166548],"[11166548, 16752820, 16154596, 16565622, 10856...",0.033333,1.000,1.000000,1.000000,1.00
2,"Show me products like ""InWeave Women Red Print...",[18921114],"[18921114, 17168254, 15322490, 15447990, 17168...",0.033333,1.000,1.000000,1.000000,1.00
3,I need product with a Woven design pattern in ...,"[12824928, 18262088]","[10451734, 18122478, 19218994, 17663018, 16719...",0.000000,0.000,0.000000,0.000000,1.00
4,I'm looking for products from MANGO for everyd...,"[15315768, 15265896, 15265898, 16892568, 15977...","[14006764, 16124612, 17607938, 18321088, 18877...",0.000000,0.000,0.000000,0.000000,1.00
5,What do you have from Sweet Dreams in Playsuit...,"[11581420, 11581366]","[11581420, 11581448, 11581366, 12882594, 13270...",0.066667,1.000,1.000000,0.919721,1.00
6,I'm looking for Kurta set by Vishudh.,"[13536726, 15997054, 18262138, 13119222, 18263...","[9867983, 13799826, 14860664, 18929144, 135367...",0.266667,1.000,0.250000,0.604468,1.00
7,I'm looking for A-line by Ancestry.,"[15407228, 19152780, 14873702]","[14873856, 15407228, 14873702, 19152780, 14873...",0.100000,1.000,0.500000,0.732829,1.00
8,"Can you find something similar to ""FASHOR Wome...",[19140952],"[19140952, 18964098, 11561972, 17487138, 18964...",0.033333,1.000,1.000000,1.000000,1.00
9,Show me Blue Tailored jacket from Darzi.,[14460680],"[14460680, 17513236, 14460654, 14460662, 17666...",0.033333,1.000,1.000000,1.000000,1.00
